# Qwen3-8B gốc, 2-shot — hai khuôn ngữ cảnh (GraphRAG + Naive RAG)

Notebook **mỏng có chủ ý**: mọi logic nằm trong `finetune/kaggle_8b_eval.py` (trong git,
có xuất xứ commit). Notebook chỉ điều phối ba ô. Vá logic bằng cell trên Kaggle
chính là chuyện đã làm số liệu phiên 1 FT-03 không khớp bất kỳ commit nào
(`gate_base_model.md` §6) — đừng lặp lại.

**Cấu hình panel phải:** Accelerator `GPU T4 x2` · Internet `On` ·
Persistence `Files only` · Visibility `Private`

**Kaggle Secret bắt buộc** (đọc qua `kaggle_secrets`, KHÔNG hardcode): chỉ `HF_TOKEN`.
Repo GitHub công khai nên clone không cần credential.

---

**Hai ô, một cấu hình sinh duy nhất:**

| Ô | Mô hình sinh | n_shot | Ngữ cảnh |
|---|---|---|---|
| 1 | Qwen3-8B gốc (Q4_K_M) | 2 | GraphRAG — `results_graphrag_final1_20260729-022916.json` |
| 2 | Qwen3-8B gốc (Q4_K_M) | 2 | Naive RAG — `results_baseline_20260710-085236.json` |

**Vì sao chỉ 2-shot.** Lượt 4B đã đo: mô hình gốc 0-shot đạt `format_ok` 6.6%,
2-shot đạt 78.1%. Với mô hình gốc, few-shot là điều kiện để kết quả có nghĩa —
chạy 0-shot cho 8B chỉ tốn giờ GPU để đo lại một thất bại đã biết.

**Vì sao dùng đúng hai file ngữ cảnh của lượt 4B.** Truy hồi đóng băng ở cả hai ô;
mỗi ô chỉ đổi mô hình sinh. Nhờ vậy Δ so với hàng 4B trong `ft06b_matrix.md` phản
ánh **đúng một biến** là cỡ mô hình, không lẫn khác biệt ngữ cảnh.

**Tham số sinh** giống hệt lượt 4B (`gate_base_model.md` §3): temperature 0.7 ·
top_p 0.8 · top_k 20 · min_p 0 · **presence_penalty 0** · seed 42 ·
max_new_tokens 2048 · n_ctx 16384 · n_gpu_layers −1. Bảy trong tám giá trị đã là
mặc định của `replay.py` nên script không truyền lại; `presence_penalty` là ngoại lệ
duy nhất phải truyền tường minh vì mặc định của `replay.py` là 1.0.

**Ghim GPU.** Cả hai ô chạy tuần tự trên `CUDA_VISIBLE_DEVICES=0` (ghim qua env của
subprocess). 8B Q4_K_M ~5 GB, n_ctx 16384 vẫn vừa một card T4 16 GB.

In [ ]:
import os,subprocess;from kaggle_secrets import UserSecretsClient as S;s=S();os.environ.update(HF_TOKEN=s.get_secret("HF_TOKEN"),HF_XET_HIGH_PERFORMANCE="1");R="/kaggle/working/repo";B="dev/fine-tune";U="https://github.com/tandat-dao/vn-legal-graphrag.git";r=subprocess.run((f"git -C {R} fetch -q --all && git -C {R} checkout -q {B} && git -C {R} pull -q --ff-only" if os.path.isdir(R+"/.git") else f"git clone -q -b {B} {U} {R}")+f" && git -C {R} log -1 --format='commit %H%n  %s'",shell=True,capture_output=True,text=True);print(r.stdout+r.stderr,">>> GHI COMMIT HASH NÀY VÀO KHÓA LUẬN")

In [ ]:
!cd /kaggle/working/repo && python finetune/kaggle_8b_eval.py --stage prep

In [ ]:
!cd /kaggle/working/repo && python finetune/kaggle_8b_eval.py --stage run && python finetune/kaggle_8b_eval.py --stage table

**Session đứt giữa ô?** Chạy lại ô 1 → ô 2, rồi ô 3. Chặng `run` truyền `--resume`
cho `replay.py`, mà `--out` là tên TẤT ĐỊNH (không timestamp) nên `.partial.jsonl`
được đọc lại đúng chỗ dở thay vì sinh lại từ câu 1. Muốn chạy đúng một ô:
`--stage run --cells 2`.

**Lần chạy đầu chưa ghim sha256 GGUF.** Repo `Qwen/Qwen3-8B-GGUF` chưa từng dùng ở
dự án này nên không có giá trị nào để đối chiếu. Chặng `prep` TÍNH sha256 rồi ghi
vào `finetune/reports/8b_artifacts.json`. **Lần chạy sau phải truyền
`--gguf-sha256 <giá trị đó>`** để biến nó thành cổng chặn thật — nếu không, file
trên Hub đổi mà số liệu vẫn chạy tiếp trong im lặng.

**Tên file trên Hub.** `prep` thử lần lượt vài biến thể hoa/thường
(`Qwen3-8B-Q4_K_M.gguf`, `qwen3-8b-q4_k_m.gguf`, …). Nếu cả ba đều trượt, mở trang
repo xem tên thật rồi truyền `--gguf-file`.

**Đọc kết quả ở đâu.** `--stage table` ghi `finetune/reports/8b_eval.md`: bốn thang
đo chính, cột Δ đối chiếu với ô 4B gốc 2-shot cùng ngữ cảnh, và bảng sức khoẻ ô.
Chú ý cột **chạm trần token** — `n_hit_token_cap > 0` nghĩa là có câu bị cắt ở 2048
token nên mất khối trích dẫn cuối câu → F1 = 0 vì lý do kỹ thuật, không phải vì mô
hình chọn sai điều khoản. 8B sinh dài hơn 4B nên đây là rủi ro cần nhìn trước khi
diễn giải bất kỳ con số nào.

**Tải kết quả về.** Hai file `finetune/results/results_{graphrag,baseline}_8b-base-s2.json`
nằm trong `/kaggle/working/repo`, không tự đẩy lên HF (khác `kaggle_ft06.py`) — nhớ
tải xuống trước khi session hết hạn.